<a href="https://colab.research.google.com/github/rishitaramola/judicial-reasoning-env/blob/main/admin_tools/JusticeEngine_RL_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU name: Tesla T4


In [3]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade trl transformers accelerate peft datasets bitsandbytes

In [4]:
import os
if not os.path.exists("/content/judicial-reasoning-env"):
    !git clone https://github.com/rishitaramola/judicial-reasoning-env.git
%cd /content/judicial-reasoning-env
!ls

Cloning into 'judicial-reasoning-env'...
remote: Enumerating objects: 265, done.
remote: Counting objects: 100% (265/265), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 265 (delta 136), reused 235 (delta 106), pack-reused 0 (from 0)
Receiving objects: 100% (265/265), 512.58 KiB | 2.66 MiB/s, done.
Resolving deltas: 100% (136/136), done.
/content/judicial-reasoning-env
data		graders       openenv.yaml    README.md		tasks
Dockerfile	inference.py  __pycache__     requirements.txt	uv.lock
environment.py	__init__.py   pyproject.toml  server


In [5]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU name: Tesla T4


In [6]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade trl transformers accelerate peft datasets bitsandbytes

In [7]:
import os

# Clone repo (skip if already cloned)
if not os.path.exists("/content/judicial-reasoning-env"):
    !git clone https://github.com/rishitaramola/judicial-reasoning-env.git

# Move into project directory
%cd /content/judicial-reasoning-env
!ls   # You should see: server/, data/, environment.py, admin_tools/, etc.

/content/judicial-reasoning-env
data		graders       openenv.yaml    README.md		tasks
Dockerfile	inference.py  __pycache__     requirements.txt	uv.lock
environment.py	__init__.py   pyproject.toml  server


In [22]:
import torch
import sys
sys.path.insert(0, "/content/judicial-reasoning-env")

from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
from environment import JudicialEnv, JudicialAction

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("✅ All imports successful — ready to train!")

✅ GPU: Tesla T4
✅ VRAM: 15.6 GB
✅ All imports successful — ready to train!


In [23]:
MODEL_NAME     = "unsloth/llama-3-8b-bnb-4bit"
MAX_SEQ_LENGTH = 4096
LORA_RANK      = 16

print("Loading model... (takes 1-2 min on first run)")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = MODEL_NAME,
    max_seq_length         = MAX_SEQ_LENGTH,
    load_in_4bit           = True,
    fast_inference         = False,
    max_lora_rank          = LORA_RANK,
    gpu_memory_utilization = 0.6,
)
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha                 = LORA_RANK,
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)
print(f"✅ Model loaded: {MODEL_NAME}")
print(f"✅ LoRA rank: {LORA_RANK} | 4-bit quantized | Ready to train!")

Loading model... (takes 1-2 min on first run)
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.6.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
[transformers] Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Model loaded: unsloth/llama-3-8b-bnb-4bit
✅ LoRA rank: 16 | 4-bit quantized | Ready to train!


In [24]:
import re, json
from environment import JudicialEnv, JudicialAction

SYSTEM_PROMPT = """You are JusticeEngine-01, an AI legal mediator for Indian courts.
Strictly follow the Constitution of India and the Bharatiya Nyaya Sanhita (BNS).
NEVER invent monetary amounts. Use only figures from the case facts.
Respond ONLY in valid XML format:
<action>
  <verdict>liable OR not_liable OR guilty OR not_guilty OR forward_to_judge</verdict>
  <confidence_score>0.9</confidence_score>
  <reasoning_chain>Your step-by-step reasoning citing specific statutes</reasoning_chain>
</action>"""

def extract_xml_action(completion):
    try:
        verdict    = re.search(r'<verdict>(.*?)</verdict>', completion, re.DOTALL)
        confidence = re.search(r'<confidence_score>(.*?)</confidence_score>', completion, re.DOTALL)
        reasoning  = re.search(r'<reasoning_chain>(.*?)</reasoning_chain>', completion, re.DOTALL)
        return {
            "verdict":          verdict.group(1).strip()           if verdict    else "invalid",
            "confidence_score": float(confidence.group(1).strip()) if confidence else 0.0,
            "reasoning_chain":  reasoning.group(1).strip()         if reasoning  else "",
            "cited_precedents": []
        }
    except Exception:
        return None

def format_reward(prompts, completions, **kwargs):
    rewards = []
    for comp in completions:
        s = comp[0]["content"] if isinstance(comp, list) else comp
        rewards.append(0.5 if "<action>" in s and "</action>" in s else 0.0)
    return rewards

def accuracy_reward(prompts, completions, **kwargs):
    rewards = []
    for comp in completions:
        s = comp[0]["content"] if isinstance(comp, list) else comp
        action_dict = extract_xml_action(s)
        if not action_dict or action_dict["verdict"] == "invalid":
            rewards.append(-1.0); continue
        try:
            action = JudicialAction(**action_dict)
            env = JudicialEnv(domain="contract", difficulty="easy")
            env.reset()
            _, reward, _, _, info = env.step(action)
            rewards.append(float(info.get("accuracy_score", 0.0)))
        except Exception:
            rewards.append(-0.5)
    return rewards

def logic_reward(prompts, completions, **kwargs):
    rewards = []
    for comp in completions:
        s = comp[0]["content"] if isinstance(comp, list) else comp
        action = extract_xml_action(s)
        if action and "reasoning_chain" in action:
            text = action["reasoning_chain"].lower()
            score = 0.0
            if "constitution" in text:              score += 0.2
            if "bns" in text or "sanhita" in text: score += 0.3
            if "specific relief" in text:           score += 0.2
            if "limitation act" in text:            score += 0.1
            if len(text) > 100:                     score += 0.2
            rewards.append(min(score, 1.0))
        else:
            rewards.append(0.0)
    return rewards

print("✅ Reward functions defined (format + accuracy + logic)")


✅ Reward functions defined (format + accuracy + logic)


In [29]:
import json, os
from datasets import Dataset

with open("data/cases.json", "r") as f:
    cases = json.load(f)

dataset_list = []
for c in cases:
    prompt_text = (
        f"FACT PATTERN:\n{c['fact_pattern']}\n\n"
        f"STATUTES:\n{chr(10).join(c.get('applicable_statutes', []))}"
    )
    dataset_list.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt_text}
        ]
    })

dataset = Dataset.from_list(dataset_list)
print(f"✅ Dataset loaded: {len(dataset)} cases")
print(f"   Sample prompt preview: {dataset[0]['prompt'][1]['content'][:100]}...")


✅ Dataset loaded: 14 cases
   Sample prompt preview: FACT PATTERN:
Ramesh agreed to supply 500 kg of wheat to Suresh by March 1st for Rs. 50,000. Ramesh ...


In [31]:
import inspect
from trl import GRPOConfig

sig = inspect.signature(GRPOConfig.__init__)
all_params = list(sig.parameters.keys())
# Show only GRPO-specific and length-related params
grpo_params = [p for p in all_params if any(k in p.lower() for k in
               ['length', 'token', 'gen', 'vllm', 'reward', 'kl', 'clip', 'num_'])]
print("GRPO-specific params available:")
for p in grpo_params:
    print(f"  - {p}")


GRPO-specific params available:
  - num_train_epochs
  - average_tokens_across_devices
  - include_num_input_tokens_seen
  - hub_token
  - dataloader_num_workers
  - length_column_name
  - num_generations
  - num_generations_eval
  - max_completion_length
  - ds3_gather_for_generation
  - generation_batch_size
  - steps_per_generation
  - generation_kwargs
  - use_vllm
  - vllm_mode
  - vllm_model_impl
  - vllm_enable_sleep_mode
  - vllm_structured_outputs_regex
  - vllm_server_base_url
  - vllm_server_host
  - vllm_server_port
  - vllm_server_timeout
  - vllm_group_port
  - vllm_gpu_memory_utilization
  - vllm_max_model_length
  - vllm_tensor_parallel_size
  - num_iterations
  - reward_weights
  - scale_rewards
  - vllm_importance_sampling_correction
  - vllm_importance_sampling_mode
  - vllm_importance_sampling_cap
  - use_bias_correction_kl
  - num_completions_to_print
  - vllm_sampling_params
  - unsloth_num_chunks


In [35]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
print(f"✅ Chat template set: {tokenizer.chat_template[:80]}...")

✅ Chat template set: {{ bos_token }}{% for message in messages %}{% if message['role'] == 'user' %}{{...


In [36]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    use_vllm                    = False,
    learning_rate               = 5e-6,
    adam_beta1                  = 0.9,
    adam_beta2                  = 0.99,
    weight_decay                = 0.1,
    warmup_steps                = 10,
    lr_scheduler_type           = "cosine",
    logging_steps               = 1,
    bf16                        = False,
    fp16                        = True,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations             = 4,
    max_completion_length       = 512,
    max_steps                   = 250,
    save_steps                  = 250,
    output_dir                  = "outputs",
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [format_reward, accuracy_reward, logic_reward],
    args             = training_args,
    train_dataset    = dataset,
)

print("🚀 Starting RL Training — 250 steps (~15-20 min on T4)...")
trainer.train()
print("✅ Training complete!")

🚀 Starting RL Training — 250 steps (~15-20 min on T4)...


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14 | Num Epochs = 18 | Total steps = 250
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
[transformers] Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward / mean,rewards / format_reward / std,rewards / accuracy_reward / mean,rewards / accuracy_reward / std,rewards / logic_reward / mean,rewards / logic_reward / std
1,0.000000,-1.000000,0.000000,154.750000,18.000000,512.000000,0.250000,35.666668,18.000000,69.000000,0.000008,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
2,0.000000,-1.000000,0.000000,314.250000,1.000000,512.000000,0.250000,248.333344,1.000000,469.000000,0.000006,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
3,0.000000,-0.625000,0.750000,360.750000,39.000000,512.000000,0.500000,209.500000,39.000000,380.000000,0.000009,0.125000,0.250000,-0.750000,0.500000,0.000000,0.000000
4,0.000000,-0.375000,1.250000,202.500000,96.000000,452.000000,0.000000,202.500000,96.000000,452.000000,0.000008,0.125000,0.250000,-0.500000,1.000000,0.000000,0.000000
5,0.000000,-0.500000,1.000000,347.750000,153.000000,512.000000,0.500000,183.500000,153.000000,214.000000,0.000007,0.000000,0.000000,-0.500000,1.000000,0.000000,0.000000
6,0.000000,-1.000000,0.000000,69.500000,1.000000,217.000000,0.000000,69.500000,1.000000,217.000000,0.000005,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
7,0.000000,-0.150000,0.994987,295.250000,37.000000,512.000000,0.500000,78.500000,37.000000,120.000000,0.000007,0.250000,0.288675,-0.500000,0.577350,0.100000,0.200000
8,0.000000,-0.450000,1.100000,149.250000,14.000000,445.000000,0.000000,149.250000,14.000000,445.000000,0.000016,0.000000,0.000000,-0.500000,1.000000,0.050000,0.100000
9,0.000000,-0.875000,0.250000,261.500000,129.000000,357.000000,0.000000,261.500000,129.000000,357.000000,0.000010,0.125000,0.250000,-1.000000,0.000000,0.000000,0.000000
10,0.000000,-1.000000,0.000000,276.500000,16.000000,512.000000,0.250000,198.000000,16.000000,464.000000,0.000025,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000


[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
[transforme

KeyboardInterrupt: 

In [37]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    use_vllm                    = False,
    learning_rate               = 5e-6,
    adam_beta1                  = 0.9,
    adam_beta2                  = 0.99,
    weight_decay                = 0.1,
    warmup_steps                = 5,
    lr_scheduler_type           = "cosine",
    logging_steps               = 1,
    bf16                        = False,
    fp16                        = True,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2,
    num_generations             = 2,       # ← reduced from 4 to 2
    max_completion_length       = 128,     # ← reduced from 512 to 128
    max_steps                   = 60,      # ← reduced from 250 to 60 (~15 min)
    save_steps                  = 60,
    output_dir                  = "outputs",
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [format_reward, accuracy_reward, logic_reward],
    args             = training_args,
    train_dataset    = dataset,
)

print("🚀 Starting FAST RL Training — 60 steps (~12-15 min on T4)...")
trainer.train()
print("✅ Training complete!")

🚀 Starting FAST RL Training — 60 steps (~12-15 min on T4)...


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWa

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward / mean,rewards / format_reward / std,rewards / accuracy_reward / mean,rewards / accuracy_reward / std,rewards / logic_reward / mean,rewards / logic_reward / std
1,0.000000,-1.000000,0.000000,92.500000,57.000000,128.000000,0.500000,57.000000,57.000000,57.000000,0.000246,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
2,0.000000,0.350000,1.909188,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000269,0.250000,0.353553,0.000000,1.414214,0.100000,0.141421
3,0.000001,-1.000000,0.000000,101.500000,75.000000,128.000000,0.500000,75.000000,75.000000,75.000000,0.000568,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
4,0.000001,-1.000000,0.000000,45.500000,18.000000,73.000000,0.000000,45.500000,18.000000,73.000000,0.000539,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
5,0.000000,-1.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000220,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
6,0.000001,-0.250000,1.060660,64.500000,1.000000,128.000000,0.500000,1.000000,1.000000,1.000000,0.001327,0.250000,0.353553,-0.500000,0.707107,0.000000,0.000000
7,0.000000,-1.000000,0.000000,81.000000,34.000000,128.000000,0.500000,34.000000,34.000000,34.000000,0.000276,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
8,0.000000,-1.000000,0.000000,112.000000,96.000000,128.000000,0.500000,96.000000,96.000000,96.000000,0.000112,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
9,0.000000,-1.000000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000046,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
10,0.000000,-1.000000,0.000000,64.500000,1.000000,128.000000,0.500000,1.000000,1.000000,1.000000,0.000173,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000


[transformers] Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
[transforme

✅ Training complete!


In [40]:
model.save_pretrained("outputs/justice_engine_lora")
tokenizer.save_pretrained("outputs/justice_engine_lora")
print("✅ LoRA adapter saved!")

import os
files = os.listdir("outputs/justice_engine_lora")
print(f"   Files: {files}")
total_size = sum(os.path.getsize(f"outputs/justice_engine_lora/{f}") for f in files)
print(f"   Total size: {total_size / 1e6:.1f} MB")

[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/justice_engine_lora/tokenizer_config.json.


✅ LoRA adapter saved!
   Files: ['README.md', 'tokenizer.json', 'chat_template.jinja', 'tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json']
   Total size: 185.1 MB
